In [0]:
/* ============================================================
   0) RUNTIME PARAMETERS (dynamic end_date)
   - end_date = last day of month, one month before the earlier of:
       max medical service_date vs max pharmacy fill_date
   ============================================================ */
CREATE OR REPLACE TEMP VIEW runtime_parameters AS
SELECT
    (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events) AS max_medical_date,
    (SELECT MAX(fill_date)    FROM com_edp_prd.com_raw.kom_pharmacy_events) AS max_pharmacy_date,
    LAST_DAY(
        ADD_MONTHS(
            LEAST(
                (SELECT MAX(service_date) FROM com_edp_prd.com_raw.kom_medical_events),
                (SELECT MAX(fill_date)    FROM com_edp_prd.com_raw.kom_pharmacy_events)
            ), -1
        )
    ) AS end_date,
    CURRENT_DATE() AS run_date;

/* Optional: sanity check */
-- SELECT * FROM runtime_parameters;



/*
PURPOSE
- Identify eligible patients based on diagnosis and treatment criteria.
- Pull all diagnosis and treatment claims for those eligible patients.

BUSINESS LOGIC SUMMARY
1) Pull diagnosis claims (E761, E763).
2) Pull treatment claims (specific NDCs and procedure codes).
3) Identify:
  - E761 patients with ≥2 diagnosis dates + at least one treatment → specified_patients.
  - E763 patients with ≥2 diagnosis dates + Elaprase treatment,
    excluding already specified patients → incremental_patients.
4) Eligible patients = specified + incremental.
5) Return all diagnosis and treatment claims for eligible patients.
*/


/* ============================================================
   1) ALL DIAGNOSIS CLAIMS (E761 / E763)
   ============================================================ */
CREATE OR REPLACE TEMPORARY VIEW all_dx_claims AS

/* Medical diagnosis claims */
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,  -- Provider attribution
    SERVICE_DATE AS FILL_DATE,
    MEDICAL_EVENT_ID AS claim_id,
    KH_PLAN_ID AS plan_id,
    'MEDICAL' AS CLAIM_SOURCE,
    'MEDICAL' AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_medical_events
WHERE (DIAGNOSIS_CODES LIKE '%E761%' OR DIAGNOSIS_CODES LIKE '%E763%')
  AND SERVICE_DATE BETWEEN DATE '2020-08-01'
                      AND (SELECT end_date FROM runtime_parameters)

UNION

/* Pharmacy diagnosis claims (paid only) */
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    PHARMACY_EVENT_ID AS claim_id,
    COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS plan_id,
    'PHARMACY' AS CLAIM_SOURCE,
    TRANSACTION_RESULT AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE DIAGNOSIS_CODE IN ('E761', 'E763')
  AND TRANSACTION_STATUS = 'PAID'
  AND FILL_DATE BETWEEN DATE '2020-08-01'
                   AND (SELECT end_date FROM runtime_parameters);



/* ============================================================
   2) ALL TREATMENT CLAIMS
   ============================================================ */
CREATE OR REPLACE TEMPORARY VIEW all_tx_claims AS

/* Medical NDC-based treatment */
SELECT DISTINCT
    PATIENT_ID,
    COALESCE(RENDERING_NPI, REFERRING_NPI) AS NPI,
    SERVICE_DATE AS FILL_DATE,
    MEDICAL_EVENT_ID AS claim_id,
    NDC11 AS CODE,
    KH_PLAN_ID AS plan_id,
    'MEDICAL' AS CLAIM_SOURCE,
    'MEDICAL' AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_medical_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND SERVICE_DATE BETWEEN DATE '2023-08-01'
                      AND (SELECT end_date FROM runtime_parameters)

UNION

/* Medical procedure-based treatment */
SELECT DISTINCT
    PATIENT_ID,
    RENDERING_NPI AS NPI,
    SERVICE_DATE AS FILL_DATE,
    MEDICAL_EVENT_ID AS claim_id,
    PROCEDURE_CODE AS CODE,
    KH_PLAN_ID AS plan_id,
    'MEDICAL' AS CLAIM_SOURCE,
    'MEDICAL' AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_medical_events
WHERE PROCEDURE_CODE IN ('99601', '99602', '96365', '96366', 'J1743',
                         'S9357', 'S9379', '38206', '38230', '38232',
                         '38240', '38241', '38242', '38243', '38250')
  AND SERVICE_DATE BETWEEN DATE '2023-08-01'
                      AND (SELECT end_date FROM runtime_parameters)

UNION

/* Pharmacy treatment (paid only) */
SELECT DISTINCT
    PATIENT_ID,
    PRESCRIBER_NPI AS NPI,
    FILL_DATE,
    PHARMACY_EVENT_ID AS claim_id,
    NDC11 AS CODE,
    COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS plan_id,
    'PHARMACY' AS CLAIM_SOURCE,
    TRANSACTION_RESULT AS TRANSACTION_STATUS
FROM com_edp_prd.com_raw.kom_pharmacy_events
WHERE NDC11 IN ('54092070001', '540920700')
  AND TRANSACTION_RESULT = 'PAID'
  AND FILL_DATE BETWEEN DATE '2023-08-01'
                   AND (SELECT end_date FROM runtime_parameters);



/* ============================================================
   3) E761 PATIENTS WITH ≥2 DIAGNOSIS DATES
   ============================================================ */
CREATE OR REPLACE TEMPORARY VIEW e761_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E761%'
      AND SERVICE_DATE BETWEEN DATE '2020-08-01'
                          AND (SELECT end_date FROM runtime_parameters)

    UNION

    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E761'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN DATE '2020-08-01'
                       AND (SELECT end_date FROM runtime_parameters)
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;



/* ============================================================
   4) SPECIFIED PATIENTS
   - E761 with ≥2 dx
   - AND at least one treatment claim
   ============================================================ */
CREATE OR REPLACE TEMPORARY VIEW specified_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e761_patients_2dx e
INNER JOIN all_tx_claims t
    ON e.PATIENT_ID = t.PATIENT_ID;



/* ============================================================
   5) E763 PATIENTS WITH ≥2 DIAGNOSIS DATES
   ============================================================ */
CREATE OR REPLACE TEMPORARY VIEW e763_patients_2dx AS
SELECT PATIENT_ID
FROM (
    SELECT DISTINCT PATIENT_ID, SERVICE_DATE AS FILL_DATE
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES LIKE '%E763%'
      AND SERVICE_DATE BETWEEN DATE '2020-08-01'
                          AND (SELECT end_date FROM runtime_parameters)

    UNION

    SELECT DISTINCT PATIENT_ID, FILL_DATE
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE = 'E763'
      AND TRANSACTION_STATUS = 'PAID'
      AND FILL_DATE BETWEEN DATE '2020-08-01'
                       AND (SELECT end_date FROM runtime_parameters)
)
GROUP BY PATIENT_ID
HAVING COUNT(DISTINCT FILL_DATE) >= 2;



/* ============================================================
   6) ELAPRASE TREATMENT PATIENTS
   ============================================================ */
CREATE OR REPLACE TEMPORARY VIEW elaprase_tx AS
SELECT DISTINCT PATIENT_ID
FROM all_tx_claims
WHERE CODE IN ('54092070001', '540920700', 'J1743');



/* ============================================================
   7) INCREMENTAL PATIENTS
   - E763 with ≥2 dx
   - AND Elaprase treatment
   - NOT already in specified_patients
   ============================================================ */
CREATE OR REPLACE TEMPORARY VIEW incremental_patients AS
SELECT DISTINCT e.PATIENT_ID
FROM e763_patients_2dx e
INNER JOIN elaprase_tx t
    ON e.PATIENT_ID = t.PATIENT_ID
WHERE e.PATIENT_ID NOT IN (
    SELECT PATIENT_ID FROM specified_patients
);



/* ============================================================
   8) ELIGIBLE PATIENTS
   ============================================================ */
CREATE OR REPLACE TEMPORARY VIEW eligible_patients AS
SELECT PATIENT_ID FROM specified_patients
UNION
SELECT PATIENT_ID FROM incremental_patients;



/* ============================================================
   9) ALL CLAIMS FOR ELIGIBLE PATIENTS
   ============================================================ */
CREATE OR REPLACE TEMPORARY VIEW all_patient_claims AS

/* Diagnosis claims */
SELECT DISTINCT
    PATIENT_ID,
    NPI,
    FILL_DATE,
    claim_id,
    plan_id,
    CLAIM_SOURCE,
    TRANSACTION_STATUS
FROM all_dx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients)

UNION

/* Treatment claims */
SELECT DISTINCT
    PATIENT_ID,
    NPI,
    FILL_DATE,
    claim_id,
    plan_id,
    CLAIM_SOURCE,
    TRANSACTION_STATUS
FROM all_tx_claims
WHERE PATIENT_ID IN (SELECT PATIENT_ID FROM eligible_patients);

In [0]:
SELECT * FROM runtime_parameters;

SELECT COUNT(*) AS n_claims, COUNT(DISTINCT patient_id) AS n_patients
FROM all_patient_claims;


In [0]:
SELECT COUNT(*) AS n FROM runtime_parameters;